# webAI-ColVec1.1 visual retrieval arm — ViDoRe V3 physics

**P1 trong `docs/retrieval_research_plan.md`**: nâng arm visual từ ColQwen2-2B (đứng cuối
leaderboard) lên một model lớp SOTA. `webAI-ColVec1.1` hiện **#1 ViDoRe V3** (8b: 64.95 /
4b: 63.90 Mean-Task). Cùng họ late-interaction/MaxSim với ColQwen2 nên là **thay thế trực tiếp**
arm visual — không phải reranker.

**⚠️ LICENSE: webAI Non-Commercial v1.0** — KHÔNG phải Apache/MIT. Dùng cho *thí nghiệm nghiên cứu*
thì OK; **không dùng được cho sản phẩm thương mại**. Nếu team cần bản thương mại, fallback là
`nvidia/nemotron-colembed-vl-4b-v2` (#9, open) hoặc `tomoro-colqwen3-embed-4b` (#12).

**Câu hỏi cần trả lời (test standalone TRƯỚC):**
- Physics/French là domain KHÓ nhất của ViDoRe V3 (SOTA chỉ 50.84 vs mean 64) và là domain yếu
  cho visual retriever (Snappy paper). Leaderboard mean KHÔNG chuyển giao thẳng.
- (a) visual-only NDCG@10 trên physics là bao nhiêu? (ColQwen2-2B chỉ 45.7, n.s.)
- (b) fused với text KDL α=0.7 (43.86) — có vượt stack miễn phí SEP+ColQwen2 (48.35)? vượt Voyage (49.23)?
- (c) một visual arm mạnh hơn có bị reranker hấp thụ như ColQwen2-2B không (§20/§22)?

**Trước khi chạy:**
1. `Runtime → Change runtime type → GPU`. **4b cần ~9-10GB** (T4 borderline, nên dùng L4/A100
   Colab Pro). **8b cần ~17GB** → bắt buộc L4/A100, hoặc load 8-bit trên T4 (cell có sẵn switch).
2. Repo private → không `git clone`. Build bundle ở máy local:
   ```bash
   cd AXIOM_DE-RD
   git archive --format=zip -o ~/Desktop/colvec_bundle.zip HEAD
   zip -r ~/Desktop/colvec_bundle.zip \
     data/raw/benchmarks/vidore_v3_physics \
     data/benchmark/vidore_v3/physics \
     data/benchmark/vidore_v3/results/physics_KDL_pool.json
   ```
   `git archive` = code đã track (~2.5MB); `zip -r` thêm 42 PDF + 3 parquet + KDL pool
   (~102MB, gitignored). Một lần upload.
3. Notebook **không gọi API** — chỉ tải model công khai từ HuggingFace + suy luận cục bộ.

**Xuất ra:** `physics_colvec_scores.npy`, `..._keys.json`, `..._qids.json` (~2-3MB) → tải về máy
local chạy `research/experiments/physics_kdl_slate.py --visual-dir data/work/vidore_physics_colvec`
để có NDCG@10 chính thức, permutation test, và full slate (SEP / rerank / stack) với ColVec thay
ColQwen2.

In [ ]:
!nvidia-smi

## 1. Upload bundle (code + data) và cài đặt

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()  # chon colvec_bundle.zip build o Buoc chuan bi
zip_name = next(iter(uploaded))

REPO = Path("/content/AXIOM_DE-RD")
REPO.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(REPO)

pdfs = list((REPO / "data/raw/benchmarks/vidore_v3_physics").glob("*.pdf"))
pool = REPO / "data/benchmark/vidore_v3/results/physics_KDL_pool.json"
assert len(pdfs) == 42, f"expected 42 PDFs, found {len(pdfs)}"
assert pool.is_file(), "physics_KDL_pool.json missing after unzip"
assert (REPO / "pyproject.toml").is_file(), "pyproject.toml missing -- git archive at repo root?"
print(f"OK: {len(pdfs)} PDFs, KDL pool present, package layout present")

In [ ]:
%cd /content/AXIOM_DE-RD
%pip install -q -e "."
%pip install -q pymupdf
# webAI-ColVec ships custom trust_remote_code modules (modeling_colqwen35_bidirection.py,
# processing_colqwen35_bidirection.py). config.json pins transformers_version 5.14.1 and the
# model_type is "qwen3_5" -- it will NOT load on an older transformers. Versions below are
# copied from the repo's own evaluation-requirements-cu128.txt.
%pip install -q -U "transformers==5.14.1" "sentence-transformers==5.6.0" "accelerate==1.14.0"
# For 8-bit loading of the 8b on a 16GB T4 (skip if you have L4/A100):
%pip install -q -U bitsandbytes
print("deps installed -- if pip warns about a version conflict, Runtime > Restart session, "
      "then continue from the next cell (files in /content survive a restart).")

## 2. GPU profile + chọn model

- **4b** (`webAI-ColVec1.1-4b`, #3 leaderboard 63.90): ~9-10GB fp16/bf16. T4 rất sát; L4/A100 thoải mái.
- **8b** (`webAI-ColVec1.1-8b`, #1 leaderboard 64.95): ~17GB bf16 → L4/A100, hoặc `LOAD_8BIT=True` trên T4.

Bắt đầu bằng **4b** (rẻ, nhanh, gần như bằng 8b). Chỉ lên 8b nếu 4b cho tín hiệu tốt và có Colab Pro.

In [ ]:
import torch

MODEL_ID  = "webAI-Official/webAI-ColVec1.1-4b"   # -> "...-8b" khi muon
LOAD_8BIT = False   # True: nen 8-bit (chi khi chay 8b tren <20GB VRAM)
DPI       = 144     # khop ColQwen2 notebook de so sanh sach; ColVec chiu duoc cao hon (1792 visual tokens)

gpu = torch.cuda.get_device_properties(0)
print(f"GPU={gpu.name}, VRAM={gpu.total_memory / 1024**3:.1f} GiB")
DTYPE = torch.bfloat16 if any(x in gpu.name for x in ("A100","H100","L4","L40","RTX 40")) else torch.float16
print(f"MODEL_ID={MODEL_ID}  dtype={DTYPE}  8bit={LOAD_8BIT}")
if "8b" in MODEL_ID and gpu.total_memory / 1024**3 < 20 and not LOAD_8BIT:
    print("!! 8b tren <20GB VRAM: dat LOAD_8BIT=True hoac doi sang 4b")

## 3. Render page images từ PDF gốc

Dùng lại `render_page_images.py` (đã dùng cho CLIP/ColQwen2).

In [ ]:
%cd /content/AXIOM_DE-RD
!python research/experiments/render_page_images.py --dpi {DPI}

In [ ]:
from pathlib import Path
images = sorted(Path("/content/AXIOM_DE-RD/data/work/vidore_physics_page_images").glob("*.png"))
print(f"{len(images)} page images @ {DPI} DPI")
assert len(images) == 1674, f"expected 1674, got {len(images)}"

## 4. Load webAI-ColVec

Đường transformers (`AutoModel` + `AutoProcessor`), cùng khuôn với ColQwen2 notebook:
`process_images` / `process_queries` / `score_retrieval`. `trust_remote_code=True` bắt buộc.

Đã đối chiếu với source thật trong repo (`modeling_colqwen35_bidirection.py`):
- class `ColQwen35Bidirection(Qwen3_5ForConditionalGeneration)` — đúng họ ColQwen, base Qwen3.5-VL,
  vá `is_causal=False` mọi lớp text (full bidirectional).
- `model(**inp)` trả **thẳng một tensor** đã L2-norm + mask (không phải ModelOutput) → `embed()` bên dưới
  xử lý cả 2 trường hợp.
- `embedding_dim=640` (config.json), scoring float32. `score_retrieval(q, d, batch_size=128,
  output_dtype=None, output_device="cpu")` — MaxSim bằng einsum theo batch, khớp phần tính tay ở Bước 6.
- query augmentation = 10 pad token (ColPali dùng nhiều hơn); `max_num_visual_tokens` default None,
  ta set 1792 đúng như eval của webAI.

In [ ]:
from transformers import AutoModel, AutoProcessor

_load_kw = dict(trust_remote_code=True, attn_implementation="sdpa", device_map="cuda:0")
if LOAD_8BIT:
    from transformers import BitsAndBytesConfig
    _load_kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
else:
    _load_kw["dtype"] = DTYPE

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_num_visual_tokens=1792)
model = AutoModel.from_pretrained(MODEL_ID, **_load_kw).eval()
print("model loaded:", sum(p.numel() for p in model.parameters()) / 1e9, "B params")


def embed(inp):
    # model(**inp) -> multi-vector tensor; robust to ModelOutput / tuple wrappers
    out = model(**inp)
    if hasattr(out, "embeddings"):
        out = out.embeddings
    elif hasattr(out, "last_hidden_state"):
        out = out.last_hidden_state
    elif isinstance(out, (tuple, list)):
        out = out[0]
    return out

## 5. Encode 1,674 trang — checkpoint, resumable

In [ ]:
import time
from PIL import Image

probe = images[:12]
t0 = time.time()
for p in probe:
    inp = processor.process_images([Image.open(p).convert("RGB")]).to(model.device)
    with torch.inference_mode():
        e = embed(inp)
per = (time.time() - t0) / len(probe)
print(f"{per:.2f}s/image -> ~{per * len(images) / 60:.1f} min cho {len(images)} anh")
print("embed shape (1 anh):", tuple(e[0].shape), " dtype:", e.dtype)

In [ ]:
import pickle

CKPT = Path("/content/colvec_page_embeddings.pkl")
page_emb = pickle.load(open(CKPT, "rb")) if CKPT.exists() else {}
if page_emb:
    print(f"resuming: {len(page_emb)}/{len(images)}")

for i, p in enumerate(images):
    key = p.stem.replace("__", "::")           # physics__File#page=N.png -> physics::File#page=N
    if key in page_emb:
        continue
    inp = processor.process_images([Image.open(p).convert("RGB")]).to(model.device)
    with torch.inference_mode():
        emb = embed(inp)[0].to(torch.float16).cpu()   # (seq_len, 640), L2-normalized
    page_emb[key] = emb
    if (i + 1) % 100 == 0:
        pickle.dump(page_emb, open(CKPT, "wb"))
        print(f"  {i + 1}/{len(images)}", flush=True)

pickle.dump(page_emb, open(CKPT, "wb"))
print(f"done: {len(page_emb)} page embeddings")

## 6. Encode 302 câu truy vấn + ma trận điểm

In [ ]:
import sys
sys.path.insert(0, "/content/AXIOM_DE-RD")
from src.evaluation.benchmarks import load

bench = load("vidore_v3", subset="physics", language="french")
qrels = bench.qrels()
questions = [q for q in bench.questions() if qrels.get(q.qid)]
print(f"{len(questions)} cau truy van co qrels")

In [ ]:
query_emb = {}
QB = 8
for s in range(0, len(questions), QB):
    chunk = questions[s:s + QB]
    inp = processor.process_queries([q.query for q in chunk]).to(model.device)
    with torch.inference_mode():
        emb = embed(inp)
    for q, e in zip(chunk, emb):
        query_emb[q.qid] = e.to(torch.float16).cpu()
print(f"done: {len(query_emb)} query embeddings  (1 truy van: {tuple(next(iter(query_emb.values())).shape)})")

In [ ]:
import numpy as np

keys = sorted(page_emb)                 # cot = trang
qids = [q.qid for q in questions]       # hang = truy van

# MaxSim tinh tay tren GPU: score(q,d) = sum_i max_j (q_i . d_j). Embeddings da L2-norm.
# Vong ngoai theo chunk tai lieu (day len GPU 1 lan), vector hoa qua toan bo truy van.
dev = "cuda"
Q = torch.nn.utils.rnn.pad_sequence(
    [query_emb[qid] for qid in qids], batch_first=True).to(dev, torch.float16)   # (Nq, Lq, 640)
qmask = torch.nn.utils.rnn.pad_sequence(
    [torch.ones(query_emb[qid].shape[0]) for qid in qids], batch_first=True).to(dev)  # (Nq, Lq)
score_matrix = np.zeros((len(qids), len(keys)), dtype=np.float32)

DCHUNK = 64
for c in range(0, len(keys), DCHUNK):
    block = keys[c:c + DCHUNK]
    for j, k in enumerate(block):
        d = page_emb[k].to(dev, torch.float16)                 # (Ld, 640)
        sim = torch.einsum("nld,md->nlm", Q, d)                # (Nq, Lq, Ld)
        per_tok = sim.max(dim=2).values * qmask                # (Nq, Lq)  padded query tokens -> 0
        score_matrix[:, c + j] = per_tok.sum(dim=1).float().cpu().numpy()
    if (c // DCHUNK) % 5 == 0:
        print(f"  scored docs {c + len(block)}/{len(keys)}", flush=True)
print("score matrix:", score_matrix.shape)

# san: neu processor co scorer rieng, so khop tren 1 truy van dau
try:
    ref = processor.score_retrieval([query_emb[qids[0]].float()], [page_emb[k].float() for k in keys[:50]])
    ref = np.asarray(ref.cpu() if hasattr(ref, "cpu") else ref).ravel()
    print("sanity vs processor.score_retrieval (top-5 corr):",
          np.corrcoef(ref, score_matrix[0, :50])[0, 1])
except Exception as e:
    print("(processor.score_retrieval khong dung duoc truc tiep, dung MaxSim tay -- OK):", type(e).__name__)

## 7. Kiểm tra nhanh trong Colab (số chính thức chạy ở local)

In [ ]:
import math

def ndcg10(ranked, gold):
    dcg = sum((2 ** gold.get(k, 0) - 1) / math.log2(i + 2) for i, k in enumerate(ranked[:10]))
    idcg = sum((2 ** g - 1) / math.log2(i + 2) for i, g in enumerate(sorted(gold.values(), reverse=True)[:10]))
    return 100 * dcg / idcg if idcg > 0 else 0.0

kx = {k: i for i, k in enumerate(keys)}
vals = [ndcg10(sorted(keys, key=lambda k: -score_matrix[r, kx[k]]), qrels[qid]) for r, qid in enumerate(qids)]
print(f"webAI-ColVec visual-only NDCG@10 = {sum(vals) / len(vals):.2f}  (n={len(vals)})")
print("so sanh: text KDL alpha0.7 = 43.86 | ColQwen2-2B visual-only = 45.70 (n.s.) | Voyage rerank = 49.23")

In [ ]:
import json as _json
pool = _json.loads(Path("/content/AXIOM_DE-RD/data/benchmark/vidore_v3/results/physics_KDL_pool.json").read_text())["queries"]

found = total = 0
for r, qid in enumerate(qids):
    if qid not in pool:
        continue
    deep = {k for k, v in qrels[qid].items() if v > 0} - set(pool[qid]["candidates"][:100])
    if not deep:
        continue
    top100 = set(sorted(keys, key=lambda k: -score_matrix[r, kx[k]])[:100])
    total += len(deep); found += len(deep & top100)
print(f"gold pages text KDL misses entirely (rank>=100): {total}")
print(f"  recovered in ColVec visual top-100: {found} ({100 * found / max(total,1):.1f}%)   "
      f"(ColQwen2-2B: 32.9%)")

## 8. Xuất về local

In [ ]:
OUT = Path("/content/physics_colvec_export"); OUT.mkdir(exist_ok=True)
np.save(OUT / "physics_colvec_scores.npy", score_matrix)
_json.dump(keys, open(OUT / "physics_colvec_keys.json", "w"))
_json.dump(qids, open(OUT / "physics_colvec_qids.json", "w"))
_json.dump({"model": MODEL_ID, "dpi": DPI, "load_8bit": LOAD_8BIT,
            "dim": int(next(iter(page_emb.values())).shape[-1]), "n_pages": len(keys), "n_queries": len(qids)},
           open(OUT / "physics_colvec_meta.json", "w"))
import shutil
shutil.make_archive("/content/physics_colvec_export", "zip", OUT)
print("exported ->", OUT)

In [ ]:
from google.colab import files
files.download("/content/physics_colvec_export.zip")

## Bước tiếp theo (máy local, không cần GPU)

1. Giải nén `physics_colvec_export.zip` vào `data/work/vidore_physics_colvec/`.
2. `python research/experiments/physics_kdl_slate.py --visual-dir data/work/vidore_physics_colvec --visual-name colvec`
   → NDCG@10 chính thức (`pytrec_eval`), permutation test vs baseline **và vs Voyage/Nemotron**,
   full slate: visual-only, + fusion sweep, + SEP, + rerank, + mọi tổ hợp.
3. So với: KDL baseline 43.86 · SEP+ColQwen2 48.35 · Nemotron 47.74 · Voyage 49.23 · SOTA physics 50.84.
4. Nếu ColVec visual-only đã ≥ ~48: thử **visual-only làm arm chính**, và kiểm tra xem reranker
   còn hấp thụ được nó không (nếu không → visual arm mạnh đã phá được pattern redundancy §20/§22).